# Sedikit cerita...

Sebenernya Resnet (Deep Residual Learning for Image Recognition) itu ga juga kek masuk di compvis specifically, wlupun disini tetep kita masukin ke folder COMPVIS karena yaa in the beginning resnet itu model yg dibuild utk ngeaddress problem di compvis (ILSVRC 2015). Kek kita tau lah skrg even di NLP khususnya transformer based itu kan pake ini, di sublayer itu kan kita msti nemu add sama norm, nah add inilah si Residual. 

> Trus kenapa kita butuh si residual ini?
Imagine we've neur network yg dalem banget (50++ layer). What we thought at bginning? exactly si model jelas bakal lebih pinter seiring bertambah dalamnya neur netnya. Yet, realitanya ga sperti itu wkwk si Pak Kaiming He dkk nemuin fenomena dimana model itu lebih dalem pretend to bodoh (errornya increase) rther than model dangkal. (In some case memang ga smua seperti itu), nah why?
- **Vanishing Gradient**, ini simpel -> kan kita ngebackprop gradient itu dengan "mengalikan" scr kontinu dg chain rule, kalau rata2 nilainya 0.1 diulang 50x jadi apa deh? yaa $0.1^{50}$, hasilnya yaa jadi 0, sekian yg deket banget sama 0. At the end layer di awal itu kaga belajar apapun
- **Degradation problem**, kalau ternyata ada suatu layer yg ga guna, model kan struggle buat identity mapping. Jadi maksain si non linear funcnya utk ngegeneratte output y=x itu susahhh 

Di arsitektur **plain network**, setiap layer ngeamapping suatu input $x$ menjadi target $f(x)$.

$$
y = f(x)
$$

Nah, pada di ResNet, insted of memaksa layer untuk mempelajari keseluruhan fungsi $f(x)$, kita kek ngasi semacam **"jalan tol" (skip connection)** bagi input $x$ agar bisa langsung diteruskan ke output.

Akibatnya, layer hanya perlu mempelajari **residual** (selisih) terhadap input, sehingga output menjadi:

$$
y = f(x) + x
$$

dengan:

- $f(x)$ = hasil yang dipelajari oleh layer (residual mapping)
- $x$ = input asli yang dilewatkan melalui skip connection
- $y$ = output akhir = hasil pembelajaran + input asli

> Nah apa hasil dari turunan output $y$ terhadap $x$?

$$
\frac{\partial y}{\partial x}
=
\frac{\partial f(x)}{\partial x}
+1
$$

Lihat angka 1 itu ☝️, ini yg jadi game changer sii resnet, kek sekecil apapun turunan si $\frac{\partial f(x)}{\partial x}$ ambilah hampir 0, gradient still ngalir ke layer sblumnya lewat angka 1 itu. No matter mo 1000 layer juga erronya tetep bakal tembus sampe balik layer depan lagi

# Import2 aje

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [2]:
# shape convention -> B itu batch size, D nya hidden dim

class PlainBlock(nn.Module):
    """
    Standard block. x masuk, diolah, dan x yang lama dibuang.
    Matematis: y = f(x)
    """
    def __init__(self,d_model):
        super().__init__()
        # inisialisasinya dbkin jadi normal dist gittu random gaussian
        self.fc = nn.Linear(d_model,d_model)
        self.act = nn.ReLU()

        # kita kecilin weight initnya, biar simulasinya tuu bisa lebi clear klo si resnet better
        nn.init.normal_(self.fc.weight,mean=0,std=0.1)
        nn.init.zeros_(self.fc.bias)

    def forward(self,x):
        # x: (B,D)
        out = self.fc(x)
        out = self.act(x)
        return out
    
class ResidualBlock(nn.Module):
    """
    Residual block. x masuk, diolah jadi f(x), lalu DITAMBAHKAN dengan x awal.
    Matematis: y = f(x) + x
    """
    def __init__(self,d_model):
        super().__init__()
        # inisialisasinya percis yaa biar fair
        self.fc = nn.Linear(d_model,d_model)
        self.act = nn.ReLU()

        nn.init.normal_(self.fc.weight,mean=0,std=0.1)
        nn.init.zeros_(self.fc.bias)

    # nah disni yg pembeda
    def forward(self,x):
        residual = x
        x = self.fc(x) 
        x = self.act(x)
        out = x+residual
        return out


In [3]:
class DeepNet(nn.Module):
    def __init__(self,block_type,d_model,n_layers):
        super().__init__()
        self.n_layers = n_layers

        # bkin dia jejer gt
        self.layers = nn.ModuleList([
            block_type(d_model) for _ in range(n_layers)
        ])
        self.head = nn.Linear(d_model,1)

    def forward(self,x):
        # x: B, D masih sm

        # kita store gradientnya di layer awal buat diinspect
        layer_outputs = []

        h = x
        for i, layer in enumerate(self.layers):
            h = layer(h)
            # Retain_grad biar Torch nyimpen gradient di intermediate layer ini
            h.retain_grad()
            layer_outputs.append(h)

        out = self.head(h) # (B,1)
        return out, layer_outputs

In [10]:
def trace_residual_magic():
    print("\n" + "=" * 68)
    print("THE MAGIC OF RESIDUAL CONNECTIONS — GRADIENT TRACING")
    print("=" * 68)

    B = 2
    D = 64
    NUM_LAYERS = 30 

    # Bikin 2 model yang sama dalamnya
    plain_net = DeepNet(PlainBlock, d_model=D, n_layers=NUM_LAYERS)
    res_net   = DeepNet(ResidualBlock, d_model=D, n_layers=NUM_LAYERS)

    # Dummy Input
    x = torch.randn(B, D,requires_grad=False)
    
    # Dummy Target
    target = torch.randn(B, 1)
    criterion = nn.MSELoss()

    # --- FORWARD PASS ---
    print("\n[FORWARD PASS SHAPE TRACE]")
    print(f"  Input x shape    : {x.shape}")
    
    # Plain Network
    torch.set_grad_enabled(True) # Tambahin ini biar aman
    out_plain, plain_hiddens = plain_net(x)
    loss_plain = criterion(out_plain, target)
    
    # Residual Network
    out_res, res_hiddens = res_net(x)
    loss_res = criterion(out_res, target)
    
    print(f"  Output out shape : {out_plain.shape} -> (B, 1)")

    # --- BACKWARD PASS ---
    loss_plain.backward()
    loss_res.backward()

    # --- GRADIENT INSPECTION ---
    print("\n" + "=" * 68)
    print("[BACKWARD PASS — MENGUKUR GRADIENT MAGNITUDE PER LAYER]")
    print("Semakin kecil magnitude, semakin model GAGAL belajar (Vanishing).")
    print("Layer 29 adalah layer paling atas (dekat output).")
    print("Layer 0  adalah layer paling bawah (dekat input).")
    print("=" * 68)
    
    print(f"\n  {'Layer ID':<10} | {'Plain Net Grad':<20} | {'Residual Net Grad':<20}")
    print("-" * 60)
    
    # Kita cek layer kelipatan 5 mundur dari belakang ke depan
    for i in range(NUM_LAYERS-1, -1, -5):
        # Ambil rata-rata absolute gradient di layer i
        grad_plain = plain_hiddens[i].grad.abs().mean().item()
        grad_res   = res_hiddens[i].grad.abs().mean().item()
        
        # Format string buat visualisasi
        plain_str = f"{grad_plain:.10f}"
        res_str   = f"{grad_res:.10f}"
        
        if grad_plain < 1e-7:
            plain_str += " (DEAD ☠️)"
            
        print(f"  Layer {i:<8} | {plain_str:<20} | {res_str:<20}")

    print("\n[KESIMPULAN]")
    print("  1. Di PlainNet, gradient di Layer 29 lumayan besar, tapi begitu sampai")
    print("     di Layer 0, nilainya udah 0.0000000000 (Vanishing Gradient).")
    print("     Model nggak akan pernah bisa pinter!")
    print("  2. Di ResidualNet, gradient mengalir dengan SANGAT SEHAT sampai ke")
    print("     Layer 0 berkat express highway `+ residual`. Inilah fondasi utama")
    print("     arsitektur Transformer!")

In [11]:
trace_residual_magic()


THE MAGIC OF RESIDUAL CONNECTIONS — GRADIENT TRACING

[FORWARD PASS SHAPE TRACE]
  Input x shape    : torch.Size([2, 64])


RuntimeError: can't retain_grad on Tensor that has requires_grad=False